# Pipeline de Encuesta: Bronze → Silver → Gold

Evaluación paso a paso del pipeline de ingeniería de datos para tabular encuestas de mercado.

| Capa   | Qué hace                                      | Herramienta |
|--------|-----------------------------------------------|-------------|
| Bronze | Lee Excel crudo, guarda en SQLite sin tocar   | pandas read_excel + SQLModel |
| Silver | Limpia prefijos ("a) "), calcula frecuencias   | pandas groupby + value_counts |
| Gold   | Enriquece con variables IA, lista para Writer | AI Agent (OpenAI/MiniMax) |
| Writer | Appenda 16 tablas en Word                     | python-docx |

In [ ]:
import sys
from pathlib import Path

# Agregar apps/api al path para importar los módulos del proyecto
API_ROOT = Path("../").resolve()
if str(API_ROOT) not in sys.path:
    sys.path.insert(0, str(API_ROOT))

ENCUESTA_PATH  = Path("../../../plantillas/excel/fase 2/Encuesta.xlsx").resolve()
TEMPLATE_PATH  = Path("../../../plantillas/word/fase 2/plan_15_generado.docx").resolve()
OUTPUT_PATH    = Path("../../../plantillas/word/fase 2/tabulacion_notebook.docx").resolve()

print(f"Encuesta : {ENCUESTA_PATH.exists()} → {ENCUESTA_PATH}")
print(f"Template : {TEMPLATE_PATH.exists()} → {TEMPLATE_PATH}")

---
## 1. BRONZE — Datos brutos

In [ ]:
import pandas as pd

# Leer Excel tal cual — SIN transformar nada
df_bronze = pd.read_excel(ENCUESTA_PATH, engine="openpyxl")
df_bronze = df_bronze.dropna(how="all").reset_index(drop=True)

print(f"Shape: {df_bronze.shape}  ({df_bronze.shape[0]} respondentes x {df_bronze.shape[1]} columnas)")
print("\nColumnas:")
for i, col in enumerate(df_bronze.columns):
    print(f"  [{i}] {col}")

In [ ]:
# Vista previa de los datos brutos (primeras 3 filas)
df_bronze.head(3)

In [ ]:
# Valores únicos de la Pregunta 1 (datos brutos, con prefijo "a) ")
col_p1 = df_bronze.columns[1]  # primera pregunta
print(f"Pregunta: {col_p1}")
print()
print(df_bronze[col_p1].value_counts())

In [ ]:
# Guardar en SQLite (Bronze layer persistente)
from services.pipeline.bronze import ingest_excel, extract_question_texts

survey_id, df_raw = ingest_excel(ENCUESTA_PATH, survey_name="Encuesta Shawarma 2024")
print(f"Survey ID en SQLite: {survey_id}")
print(f"Filas almacenadas: {len(df_raw)}")

question_texts = extract_question_texts(df_raw)
print(f"\nPreguntas detectadas: {len(question_texts)}")
for num, texto in list(question_texts.items())[:3]:
    print(f"  P{num}: {texto}")

---
## 2. SILVER — Limpieza + Frecuencias

In [ ]:
from services.pipeline.silver import clean_label, build_silver

# Demostración de clean_label
ejemplos = ["a) 18 a 30 Años", "b) Masculino", "c) Shawarma De Carne De Res", "Sin prefijo"]
print("Limpieza de prefijos:")
for e in ejemplos:
    print(f"  '{e}' → '{clean_label(e)}'")

In [ ]:
# Construir Silver completo
silver = build_silver(df_raw)

print(f"Total respondentes: {silver['total_responses']}")
print(f"Preguntas procesadas: {len(silver['questions'])}")

In [ ]:
# Vista de Silver para Pregunta 1
q1 = silver["questions"][1]
print(f"Pregunta 1: {q1['texto_pregunta']}")
print(f"Respondentes válidos: {q1['total_valid']}")
print()
q1["df_freq"]

In [ ]:
# Comparar todas las preguntas — resumen
resumen = []
for q_num, q_data in silver["questions"].items():
    resumen.append({
        "pregunta": q_num,
        "texto": q_data["texto_pregunta"][:50],
        "opciones": len(q_data["df_freq"]),
        "n_validos": q_data["total_valid"],
    })
pd.DataFrame(resumen)

---
## 3. GOLD — Enriquecimiento con IA + Datos listos

In [ ]:
from services.ai_agent import get_ai_agent

# Inferir nombres de variables con IA
agent = get_ai_agent()
variable_names = agent.infer_variable_names(question_texts)

print("Variables inferidas por IA:")
for num, nombre in sorted(variable_names.items()):
    print(f"  P{num}: {nombre}")

In [ ]:
from services.pipeline.gold import build_gold

gold = build_gold(silver, variable_names, survey_id=survey_id)

# Vista Gold de Pregunta 1
g1 = gold[1]
print(f"Variable: {g1['variable']}")
print(f"Total: {g1['total']}")
print()
pd.DataFrame(g1["options"])

---
## 4. WRITER — Generar documento Word

In [ ]:
from services.pipeline.writer import write_survey_to_word

output = write_survey_to_word(TEMPLATE_PATH, gold, OUTPUT_PATH)
print(f"Documento generado: {output}")
print(f"Existe: {output.exists()}")

---
## 5. Consulta SQLite — Verificar Bronze persistido

In [ ]:
import sqlite3

DB_PATH = API_ROOT / "data" / "plan.db"
con = sqlite3.connect(DB_PATH)

# Surveys registrados
df_surveys = pd.read_sql("SELECT * FROM survey ORDER BY id DESC LIMIT 5", con)
display(df_surveys)

# Muestra de respuestas brutas en SQLite
df_resp = pd.read_sql(
    f"SELECT * FROM surveyresponse WHERE survey_id={survey_id} LIMIT 20", con
)
display(df_resp)

con.close()

In [ ]:
# Frecuencias directo desde SQLite (equivalente al groupby de Silver)
con = sqlite3.connect(DB_PATH)
df_freq_sql = pd.read_sql(
    f"""
    SELECT pregunta_num, valor_raw, COUNT(*) as frecuencia
    FROM surveyresponse
    WHERE survey_id = {survey_id} AND pregunta_num = 1
    GROUP BY pregunta_num, valor_raw
    ORDER BY frecuencia DESC
    """,
    con,
)
con.close()
df_freq_sql